## Overview

This notebook is intended to:
* Mirror existing TGI functionalities in a notebook format.
* Demonstrate a use case: Transformer Asset Management.

For more details about transformer asset management, particularly as it relates to outages, and how it can be analyzed using Awesense's platform, please refer to the [UC24 - Analysis of Planned Outage for Assets Upgrades](https://github.com/Awesense/edm-app-examples/blob/master/use_cases/usecase_descriptions/UC24%20-%20Analysis%20of%20Planned%20Outage%20for%20Assets%20Upgrades.pdf) document.

## Set up

In [1]:
import getpass
import pandas as pd
import numpy as np
import plotly.express as px
import folium
import os
from dotenv import load_dotenv
import psycopg2

pd.set_option("display.max_columns", None)

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
# Checks for Google Colab: If detected,
# it bypasses reading credentials from local files and securely draws from Colab Secrets.
try:
    from google.colab import userdata
    from google.colab.userdata import SecretNotFoundError
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from contextlib import suppress
    print('☁️ Running in Google Colab. Using Colab Secrets for SQLAPI connection')
    # Look for Colab Secrets, fall back to interactive prompts if missing
    with suppress(SecretNotFoundError): os.environ['EDM_HOST'] = userdata.get('EDM_HOST')
    with suppress(SecretNotFoundError): os.environ['EDM_USER'] = userdata.get('EDM_USER')
    with suppress(SecretNotFoundError): os.environ['EDM_PASSWORD'] = userdata.get('EDM_PASSWORD')
else:
    # If running locally, loads connection parameters from a `.env` file (in this directory or parent directories)
    load_dotenv()

# Prompt the user to manually enter any missing connection parameters not found in `.env` file / Google Colab Secrets
if 'EDM_HOST' not in os.environ: os.environ['EDM_HOST'] = input('EDM server address: ').strip()
if 'EDM_USER' not in os.environ: os.environ['EDM_USER'] = input('EDM username: ')

# psycopg2/libpq automatically read the PG* environment variables, so map the shared EDM_* names onto them.
# (The password is only mapped if provided via `.env`/Colab; otherwise libpq falls back to `~/.pgpass`.)
os.environ['PGHOST'] = os.environ['EDM_HOST']
os.environ['PGUSER'] = os.environ['EDM_USER']
os.environ.setdefault('PGDATABASE', 'edm')
if 'EDM_PASSWORD' in os.environ: os.environ['PGPASSWORD'] = os.environ['EDM_PASSWORD']

print('Verifying SQLAPI connection parameters/credentials with connection attempt')

try:
    # Test database connection
    conn = psycopg2.connect('')
    conn.close()
    print('✅ SQLAPI connection parameters/credentials verified')
except psycopg2.OperationalError:
    print('⚠️ SQLAPI credentials check failed (assume missing/incorrect password, but double-check `.env` / secrets!)')
    # Prompt user securely for password
    os.environ['PGPASSWORD'] = getpass.getpass('Enter EDM Password manually: ')

# Keeps `%sql` result tables rendering correctly on newer versions of prettytable
import prettytable
if 'DEFAULT' not in vars(prettytable): prettytable.DEFAULT = prettytable.TableStyle.DEFAULT

# Load the SQL extension and connect
get_ipython().run_line_magic('reload_ext', 'sql')
get_ipython().run_line_magic('sql', 'postgresql://')

Verifying SQLAPI connection parameters/credentials with connection attempt


✅ SQLAPI connection parameters/credentials verified


## Example

The total count of meters downstream of any grid element can be found using the `grid_get_downstream()` function. The equivalent feature in TGI is the `Down` trace option.

We'll use a grid element ID `line_segment_5` in the `awefice` grid.

In [3]:
# PARAMETERS (Cell Tag: parameters)
# Do not split this cell structure. Papermill injects automated
# runtime overrides immediately below this block.
# If you don't want to be asked for input for these parameters, you can set them here.
grid_id = None  # Target grid identifier (e.g., 'awefice')
grid_element_id = None  # Target grid element (e.g., 'line_segment_5')

In [4]:
# Short-circuit check: If Papermill didn't inject values, prompt the user.
# To skip these prompts, set the parameter values directly in the parameters cell above (PAPERMILL PARAMETERS).
grid_id = grid_id or input("Enter grid ID: ").strip()  # awefice
grid_element_id = (
    grid_element_id or input("Enter grid element ID: ").strip()
)  # line_segment_5

Check when the grid was last updated.

In [5]:
%%sql

SELECT last_updated
FROM grid
WHERE grid_id = :grid_id;

 * postgresql://


1 rows affected.


last_updated
2021-01-01 20:00:00+00:00


In [6]:
%%sql

SELECT COUNT(*)
FROM grid_get_downstream(:grid_id,:grid_element_id)
WHERE type = 'Meter';

 * postgresql://


1 rows affected.


count
15


*Note*: The `grid_id` for grids can be found in the `grid` view, and the `grid_element_id` for grid elements can be found in the `grid_element` view.

**Use Case: Transformer Asset Management Outage**

Let's take a look at the transformers downstream of the grid element ID `line_segment_5` in the `awefice` grid.

In [7]:
# Get the transformer information.
result = %sql SELECT *, \
                ST_Y(geometry) as geo_latitude, \
                ST_X(geometry) as geo_longitude \
            FROM grid_get_downstream(:grid_id,:grid_element_id) \
            WHERE type LIKE 'Transformer';

# Turn the query result into a dataframe to work easily in Python.
df = result.DataFrame()

# Pull out the information from `meta` column saved as JSONB.
df = pd.concat([df.drop(['meta'], axis=1),
                df['meta'].apply(pd.Series)],
               axis=1)

# Return the first few rows.
df.head()

 * postgresql://


6 rows affected.


,grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,geo_latitude,geo_longitude,name,label,latitude,load_loss,longitude,mv_feeder,ownership,make_model,rating_kva,enclosure_id,no_load_loss,voltage_level,commission_date,primary_voltage,maintenance_type,manufacture_date,installation_type,secondary_voltage,hvmv_parent_element,is_operation_status,parent_transformer_id,winding_configuration
0,awefice,transformer_16,Transformer,transformer,A,False,False,False,False,None,con_32,con_34,both,NaN,0101000020E6100000FBE59315C3C65EC00CF5264C39A2...,49.267374,-123.105657,tr_16,tr_16,None,240,None,None,utility,None,5.0,None,47,MV/LV,1989-01-01,12500,oil,None,pole,240,None,True,None,None
1,awefice,transformer_26,Transformer,transformer,C,False,False,False,False,None,con_62,con_65,both,NaN,0101000020E610000083DC4598A2C65EC0EF130F4138A2...,49.267342,-123.103674,tr_26,tr_26,None,312,None,None,utility,None,10.0,None,58,MV/LV,1969-01-01,12500,oil,None,pole,240,None,True,None,None
2,awefice,transformer_31,Transformer,transformer,B,False,False,False,False,None,con_84,con_86,1to2,switchable_element_30,0101000020E61000002B508BC1C3C65EC09C3C3CB62BA2...,49.266959,-123.105698,tr_31,tr_31,None,600,None,None,utility,None,30.0,None,99,MV/LV,1979-01-01,12500,oil,None,pole,120,None,True,None,None
3,awefice,transformer_36,Transformer,transformer,ABC,False,False,False,False,None,con_106,con_110,both,NaN,0101000020E61000004D81CCCEA2C65EC0BA9955C82AA2...,49.266931,-123.103687,tr_36,tr_36,None,1547,None,None,utility,None,50.0,None,236,MV/LV,1989-01-01,12500,oil,None,pole,240,None,True,None,None
4,awefice,transformer_43,Transformer,transformer,ABC,False,False,False,False,None,con_141,con_143,both,NaN,0101000020E61000002463B5F97FC65EC0843DD45337A2...,49.267313,-123.101561,tr_43,tr_43,None,1854,None,None,private,None,75.0,None,267,MV/LV,2010-01-01,12500,oil,None,pole,480,None,True,None,None


We can easily visualize the data, such as the breakdown of transformers' phases, their ages, etc.

In [8]:
# Create a dataframe of counts by phases.
df_phases_cts = (
    df[["grid_element_id", "phases"]]
    .groupby("phases")
    .count()
    .reset_index()
    .rename(columns={"grid_element_id": "count"})
)

# Create a pie chart.
fig = px.pie(
    df_phases_cts,
    values="count",
    names="phases",
    title="Division of Transformers Downstream of " + grid_element_id,
)

fig.update_traces(textposition="inside", textinfo="percent+label+value")

In [9]:
# Calculate age as the number of years passed since commission date to today.
df["age"] = (
    (pd.Timestamp.now() - pd.to_datetime(df["commission_date"]))
    / np.timedelta64(1, "D")
    / 365
)

# Create a histogram of transformer age.
fig = px.histogram(df["age"], x="age", title="Transformer Age Distribution")

fig.update_layout(xaxis_title_text="Age (years)", yaxis_title_text="Count", bargap=0.2)

From the histogram above, we can see that there are 2 transformers that are 40 years of age or greater. This can also be identified by filtering out the data directly as the below.

In [10]:
# Filter the dataset for transformers with 40 years of age or greater.
df2 = df[df["age"] >= 40]

# Return the number of rows.
df2.shape[0]

2

Let's see where these transformers are on the map.

In [11]:
# Configure coordinates to display on the map.
map_latitude = df["geo_latitude"].mean()
map_longitude = df["geo_longitude"].mean()

# Create a folium TileLayer using custom map tile server
custom_tiles_layer = folium.TileLayer(
    tiles="https://{s}.tile.awesense.com/{z}/{x}/{y}.png",
    subdomains=["d", "e", "f"],
    attr='Map data from <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>',
)
m = folium.Map(
    location=[map_latitude, map_longitude],
    tiles=custom_tiles_layer,
    zoom_start=12,
)

for _, row in df2.iterrows():
    folium.Marker(
        location=[row["geo_latitude"], row["geo_longitude"]],
        tooltip=round(row["age"], 1),
    ).add_to(m)

display(m)

We can easily drill into the details of those transformers. For example, let's take a look at their KVA ratings.

In [12]:
# Create a histogram of KVA ratings.
fig = px.histogram(df2["rating_kva"], x="rating_kva", nbins=10)

fig.update_layout(
    title_text="Transformer kVA distribution",
    xaxis_title_text="Rating [kVA]",
    yaxis_title_text="Count",
    bargap=0.2,
)

fig.show()

---